In [ ]:
from arcgis import GIS

item = GIS().content.get("85d0ca4ea1ca4b9abf0c51b9bd34de2e")
flayer = item.layers[0]
df = flayer.query(where="AGE_45_54 < 1500").sdf
gis = GIS()
m3 = gis.map('Reno, NV', zoomlevel=4)
df.spatial.plot(kind='map', map_widget = m3,cmap='jet',
        renderer_type='u', # specify the unique value renderer using its notation 'u'
        col='ST'  # column to get unique values from
       )
m3

### Supported renderers
The renderering options below are documented in further detail in the [`Spatial DatFrame Reference`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#spatialdataframe). At a high level, you have the following renderers:

| Renderer     	| Syntax 	| Explanation                                                                                 	|
|--------------	|--------	|---------------------------------------------------------------------------------------------	|
| Simple       	| 's'    	| renders using one symbol only (the examples in [quickstart](#quickstart) above)             	|
| Unique       	| 'u'    	| renders each unique value with a different symbol. Suitable for categorical columns         	|
| Class breaks 	| 'c'    	| renders each group of values with a different color or size. Suitable for numerical columns 	|
| Heatmap      	| 'h'    	| renders density of point data as raster of varying colors                                   	|

### Unique value renderer
Let us explore the major cities census data further and explore if there are any categorical columns

In [ ]:
df.columns

The `ST` column contains state names in abbreviation is a good candidate. We could symbolize the points such that all which fall under the same state get the same symbol:

In [ ]:
uvi = m3.layers[0].properties.layerDefinition.drawingInfo.renderer.uniqueValueInfos
len(uvi)

In [ ]:
uvi[0]

In [ ]:
for u in uvi:
    print(u['symbol']['color'])

In [ ]:
df[df['POP_CLASS']<6]['NAME']

In [ ]:
df['ST'].value_counts()

### Color Maps and Colors

Color specifications can be input to the `plot` method in the `colors` paramater as:
 * a string representing [named colors](https://matplotlib.org/examples/color/named_colors.html)
 * an array of [RGB](http://www.tomjewett.com/colors/rgb.html) values
 * a named [color ramp](https://matplotlib.org/examples/color/colormaps_reference.html). 

#### Color Array

RGB and Alpha values can be used to create colors for symbols called in the `plot` method.  RGB stands for red, green, and blue respectively. Each RGB value is a value between 0-255, and the alpha value is a number between 0-255.

**Example to produce :**

    color = [255,0,100,1]

The above example produces a purplish color. Many websites provide details about using colors. For example, see [here](https://www.rapidtables.com/web/color/RGB_Color.html) for a color codes chart.

#### Color Maps

A color map is a collection of string values that can be given to generate a series of related colors from a defined set.

Color maps can be viewed here: https://matplotlib.org/examples/color/colormaps_reference.html

#### Color Map Helpers

To better understand the syntax for each input type, the ArcGIS API for Python provides some helper functions:

In [ ]:
from arcgis.mapping import display_colormaps

The **display_colormaps** function provides a quick, easy way to visualize the pre-defined set of colormaps you can use.  

In [ ]:
display_colormaps()

![named color ramps](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_display_colormaps.png)

You can retrieve a list of colormaps as well:

In [ ]:
from arcgis.mapping import symbol

colormaps = symbol.ALLOWED_CMAPS
for a,b,c,d,e in zip(colormaps[::5], colormaps[1::5], colormaps[2::5], colormaps[3::5], colormaps[4::5]):
    print("{:<20}{:<20}{:<20}{:<20}{:<}".format(a,b,c,d,e))

You can enter a list of color ramp names as input to the `display_colormaps` function to filter the output:

In [ ]:
display_colormaps(['Greens_r', 'PRGn', 'Dark2', 'Set1'])

![selected color ramps](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_selected_colormaps.png)

### Renderers
[`Renderers`](https://developers.arcgis.com/documentation/common-data-types/renderer-objects.htm) define how to visually represent a `feature layer` by defining [`symbols`](https://developers.arcgis.com/documentation/common-data-types/symbol-objects.htm) to represent individual features. The SDF provides you with functionality to control the way features appear by choosing the `symbol` the renderer uses.

Previous versions of the ArcGIS API for Python provided a method to specify a renderer manually, but you had to know details about the renderer before you drew your data. The [`map.add_layer()`]() method did not provide access to all options avialble for rendering datasets. The new visualization capabilities provided by the SDF allow you to draw spatial data quickly and easily with access to more rendering options.

#### Supported Renderers
The renderering options below are documented in further detail in the [`Spatial DatFrame Reference`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#spatialdataframe):

+ Simple - renders using one symbol only
+ Unique - renders on one or more string attributes
+ Class Breaks - renders on numeric data
+ Heatmap- renders point data into raster visualization

#### Renderer Syntax

+ 's' - is a simple renderer that uses one symbol only.
+ 'u' - unique renderer symbolizes features based on one or more matching string attributes.
+ 'c' - A class breaks renderer symbolizes based on the value of some numeric attribute.
+ 'h' - heatmap renders point data into a raster visualization that emphasizes areas of higher density or weighted values.

#### Using Renderers

Renderers are generated when on Spatial DataFrames when the [`plot`](https://esri.github.io/arcgis-python-api/apidoc/html/arcgis.features.toc.html#arcgis.features.SpatialDataFrame.plot) method is called.

### Symbology for Simple Renderers

The ArcGIS API for Python provides you the ability to set symbol types so you control data appearance. The  [`show_styles`]() function in the `arcgis.mapping` module assists developers with the syntax to define symbols.

#### Getting the Symbol Style


In [ ]:
from arcgis.mapping import show_styles

In [ ]:
# get the styles that are relevant to the current geometry type (points)
show_styles(df.geometry_type)

In [ ]:
m = GIS().map('United States', 4)
m

![map of US squares](http://esri.github.io/arcgis-python-api/notebooks/nbimages/13_4_map_dfplot2.png)

In [ ]:
m.center = [39, -98]
df.plot(map_widget=m,
        kind='map',
        symbol_type='simple',
        symbol_style='s',
        cmap='Greens_r',
        cstep=35,
        outline_color='binary',
        marker_size=5,
        line_width=.5,)